Elastic Net



In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
import hashlib
from sklearn.linear_model import ElasticNet
from sklearn.decomposition import PCA

In [ ]:
from google.colab import drive
# 1. Collega Drive
drive.mount('/content/drive')

# 2. Vai nella cartella dove hai i file e la cartella audio
%cd "/content/drive/MyDrive/Magistrale/Tesi/Fase2"

Mounted at /content/drive
/content/drive/MyDrive/Magistrale/Tesi/Fase2


In [ ]:
SEEDS = [42, 8, 1291, 64207, 305, 91876, 12, 456, 33920, 7]

In [ ]:
CSV_INPUT_PATH = f"audio_dataset.csv"
LABEL_COLUMN="portata"
NUMBER_OF_COMBINATIONS=50
MAIN_METRIC="MAE"
if MAIN_METRIC=="MSE":
  base_score_metric='neg_mean_squared_error'
else:
  base_score_metric='neg_mean_absolute_error'
START_FROM_SEED_INDEX=0 #0

In [ ]:
# 1. CARICAMENTO DATI
# Assumiamo che il file CSV abbia la colonna identificativa come prima colonna
df = pd.read_csv(CSV_INPUT_PATH, index_col=0)

In [ ]:
# 2. PREPARAZIONE X e y
target_column = 'portata'
X = df.drop(columns=[target_column])
y = df[target_column]

OTTIMIZZAZIONE DEGLI IPERPARAMETRI

In [ ]:
for i, seed in enumerate(SEEDS[START_FROM_SEED_INDEX:], start=START_FROM_SEED_INDEX):

  if START_FROM_SEED_INDEX != 0:
      print(f"Skipping the first {START_FROM_SEED_INDEX} seeds...")

  print(f"\n\n=== INIZIO RANDOMIZED SEARCH CON SEME {seed} ({i+1}/{len(SEEDS)}) ===\n")

  # 3. SPLIT TRAIN/TEST
  # Dividiamo i dati: 80% training, 20% test
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)

  train_indices = df.index.get_indexer(X_train.index).tolist()
  test_indices = df.index.get_indexer(X_test.index).tolist()
  print("SHA256 Train set: "+hashlib.sha256(str(train_indices).encode('utf-8')).hexdigest())
  print("SHA256 Test set: "+hashlib.sha256(str(test_indices).encode('utf-8')).hexdigest())

  scaler = StandardScaler()
  scaler.set_output(transform="pandas")
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)

  pca = PCA(n_components=0.90, random_state=seed)
  pca.set_output(transform="pandas")

  print(f"Feature originali: {X_train.shape[1]}")

  # Trasforma i dati (X_train e X_test erano già stati scalati col tuo scaler)
  X_train = pca.fit_transform(X_train)
  X_test = pca.transform(X_test)

  print(f"Feature dopo PCA: {X_train.shape[1]}")

  elastic_net_to_optimize = ElasticNet(random_state=seed)

  # 3. Parametri per la ricerca
  # alpha: forza della regolarizzazione
  # l1_ratio: 1 = Lasso (L1), 0 = Ridge (L2), in mezzo = mix dei due
  param_dist= {
      'alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
      'l1_ratio': [0.0, 0.2, 0.5, 0.8, 1.0]
  }

  randomized_search = RandomizedSearchCV(
      estimator=elastic_net_to_optimize,
      param_distributions=param_dist,
      n_iter=NUMBER_OF_COMBINATIONS,
      cv=3,
      scoring=base_score_metric,
      refit=False,
      verbose=2,
      random_state=42,
  )

  randomized_search.fit(X_train, y_train)
  print(f"Randomized Search completato con seme {seed}")

  # Risultati
  print("Migliori parametri individuati:")
  print(randomized_search.best_params_)

  results = pd.DataFrame(randomized_search.cv_results_)
  results = results.sort_values(by="rank_test_score", ascending=True)
  RESULTS_CSV_PATH=f"Results_randomized_search_elasticnet_{seed}.csv"
  results.to_csv(RESULTS_CSV_PATH, index=False)




=== INIZIO RANDOMIZED SEARCH CON SEME 42 (1/10) ===

SHA256 Train set: 72d2e222075a8f8f1af5150136c80e70a1597099b54f860bb98ff9aa9b9de413
SHA256 Test set: c942b7fdd659fe737f68c6b07db1ce15dee591ea6dc65f83c9d58ac3038fa68c
Feature originali: 1206
Feature dopo PCA: 159
Fitting 3 folds for each of 25 candidates, totalling 75 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 25 is smaller than n_iter=50. Running 25 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.016e+01, tolerance: 1.433e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[CV] END ..........................alpha=0.001, l1_ratio=0.0; total time=   1.2s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.080e+01, tolerance: 1.433e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[CV] END ..........................alpha=0.001, l1_ratio=0.0; total time=   1.1s


...

[CV] END ...........................alpha=10.0, l1_ratio=0.0; total time=   1.1s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.031e+01, tolerance: 1.420e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[CV] END ...........................alpha=10.0, l1_ratio=0.0; total time=   1.1s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.083e+01, tolerance: 1.406e-02 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


[CV] END ...........................alpha=10.0, l1_ratio=0.0; total time=   1.1s
[CV] END ...........................alpha=10.0, l1_ratio=0.2; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=0.2; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=0.2; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=0.5; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=0.5; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=0.5; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=0.8; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=0.8; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=0.8; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=1.0; total time=   0.0s
[CV] END ...........................alpha=10.0, l1_ratio=1.0; total time=   0.0s
[CV] END ...................


DEFINIZIONE DEL MODELLO FINALE

In [ ]:
'''
# 4. Creazione del miglior modello manuale
elastic_net_regressor = ElasticNet(**randomized_search.best_params_)
elastic_net_regressor.fit(X_train, y_train)
print("Addestramento finale completato")
'''

'\n# 4. Creazione del miglior modello manuale\nelastic_net_regressor = ElasticNet(**randomized_search.best_params_)\nelastic_net_regressor.fit(X_train, y_train)\nprint("Addestramento finale completato")\n'

VALUTAZIONE DEL MODELLO FINALE

In [ ]:
'''
# 3. PREDIZIONE E VALUTAZIONE
y_pred = elastic_net_regressor.predict(X_test)
'''

'\n# 3. PREDIZIONE E VALUTAZIONE\ny_pred = elastic_net_regressor.predict(X_test)\n'

In [ ]:
'''
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
r2 = r2_score(y_test, y_pred)
mse=mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"--- PERFORMANCE ELASTIC NET ---")
print(f"R^2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
'''

'\nmae = mean_absolute_error(y_test, y_pred)\nmape = mean_absolute_percentage_error(y_test, y_pred) * 100\nr2 = r2_score(y_test, y_pred)\nmse=mean_squared_error(y_test, y_pred)\nrmse = np.sqrt(mse)\n\nprint(f"--- PERFORMANCE ELASTIC NET ---")\nprint(f"R^2 Score: {r2:.4f}")\nprint(f"MAE: {mae:.4f}")\nprint(f"MAPE: {mape:.2f}%")\nprint(f"MSE: {mse:.4f}")\nprint(f"RMSE: {rmse:.4f}")\n'

In [ ]:
'''
importances = pd.Series(elastic_net_regressor.coef_, index=X_train.columns).abs()
'''

'\nimportances = pd.Series(elastic_net_regressor.coef_, index=X_train.columns).abs()\n'

In [ ]:
'''
plt.figure(figsize=(10, 6))
importances.nlargest(10).sort_values(ascending=True).plot(kind='barh', color='skyblue')

plt.title("Top 10 Feature per la stima (ElasticNet)")
plt.xlabel("Peso del Coefficiente (Impatto Assoluto)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
'''

'\nplt.figure(figsize=(10, 6))\nimportances.nlargest(10).sort_values(ascending=True).plot(kind=\'barh\', color=\'skyblue\')\n\nplt.title("Top 10 Feature per la stima (ElasticNet)")\nplt.xlabel("Peso del Coefficiente (Impatto Assoluto)")\nplt.grid(axis=\'x\', linestyle=\'--\', alpha=0.7)\nplt.tight_layout()\nplt.show()\n'